# 00.2 — Set up your environment

This notebook gets your machine ready to run every cell in the rest of the curriculum. It covers:

* Installing Python (if you don't have it).
* Creating an isolated **virtual environment** so this project doesn't conflict with other Python work.
* Installing the `neural_data_decoding` package in editable mode.
* Verifying everything works with a hello-world cell.

**Prerequisites:** a working shell (Terminal on macOS, GNOME Terminal on Linux, PowerShell or Windows Terminal on Windows).

## Section 1 — What MATLAB does

MATLAB has no equivalent of a virtual environment. You install MATLAB, you install toolboxes, and that's the entire setup. The MATLAB path tells the interpreter where to find your `.m` files.

The closest MATLAB analog: adding a directory to the MATLAB path with `addpath(genpath('/home/gerritcg/My_MATLAB'))`. That made every `.m` file under that directory callable from any other `.m` file. **Python does not work like this.** Every project gets its own isolated environment with its own dependencies, so version mismatches between projects can't break each other.

## Section 2 — The Python concepts you need

### Python interpreters

Your machine can have multiple Python installations. macOS ships with one, Linux distros ship with one, and you can install more (recommended, because system Pythons are often locked to ancient versions). To check what you have:

```bash
which python3
python3 --version
```

**This project requires Python 3.10 or later, and is developed and tested on 3.11.** If `python3 --version` shows `3.9` or lower, install a newer one — see below (pick 3.11 to match the project's own environment).

### Virtual environments

A **virtual environment** (or *venv*) is an isolated copy of Python and its packages. Each project gets its own venv so installing one project's dependencies can't break another's. This is **the modern Python convention** — every Python tutorial published in the last decade assumes you use one.

Creating a venv:

```bash
python3 -m venv .venv      # creates .venv/ in the current directory
source .venv/bin/activate  # activates it — prompt now shows (.venv)
```

Activating means your shell now uses `.venv/bin/python` instead of the system Python. `pip install <pkg>` puts packages into `.venv/lib/python3.11/site-packages/`. Deactivate with `deactivate` (yes, just type that).

### `pip install -e .` — editable installs

When you `pip install some_package` Python downloads the package from PyPI and copies it into your venv's `site-packages`. That works for libraries you don't plan to modify.

For projects you're actively developing — including this one — you want **editable** installs: `pip install -e .` from the project root. The venv now imports `neural_data_decoding` from your source tree directly, so when you save a `.py` file and restart your Jupyter kernel, the new code is picked up automatically. No re-install needed.

## Section 3 — The `neural_data_decoding` setup

Run these in your shell (NOT in this notebook — your shell needs to set up the environment *before* any editor launches):

```bash
# 1. Clone the repo (you've probably done this already if you're reading this).
git clone git@github.com:cgerrity/neural_data_decoding.git
cd neural_data_decoding

# 2. Create the venv. The project's tooling expects it at <repo>/.venv/.
python3 -m venv .venv
source .venv/bin/activate

# 3. Install the project in editable mode WITH the dev extras.
#    The [dev] extras pull in the tools the curriculum uses:
#    pytest, jupyter, ipykernel, matplotlib, nbstripout.
pip install -e ".[dev]"
```

**Now choose your editor.** Notebook [00.1 welcome — Section 2](00.1_welcome.ipynb#Section-2-—-How-to-actually-run-this-notebook) walks through three options in detail:

* **VS Code** — install the Python + Jupyter extensions, open this notebook, click **Select Kernel → `.venv/bin/python`**.
* **JupyterLab** — `pip install jupyterlab` then `jupyter lab notebooks/`.
* **Classic Jupyter Notebook** — `pip install notebook` then `jupyter notebook notebooks/`.

Whichever you pick, open this very file (`00.2_set_up_your_environment.ipynb`) and run the cells below to verify everything works.

### Verification cells

Each cell below tests one thing. If a cell errors, the message under it tells you what to fix. Make sure your editor has attached the venv's Python as the kernel before running — `import sys; print(sys.executable)` should end in `.venv/bin/python` (or `.venv\\Scripts\\python.exe` on Windows).

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), (
    "Python 3.10+ required (the project develops and tests on 3.11). "
    "Re-create the venv with a newer interpreter."
)
print("  ✓ Python version OK")

In [ ]:
import neural_data_decoding
print(f"neural_data_decoding: {neural_data_decoding.__version__}")
print("  ✓ Package imports")

In [ ]:
import torch
print(f"torch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"  ✓ CUDA available — {torch.cuda.device_count()} GPU(s) detected")
elif torch.backends.mps.is_available():
    print("  ✓ MPS (Apple Silicon GPU) available")
else:
    print("  ✓ CPU-only (fine for the curriculum; training takes longer)")

In [ ]:
from pathlib import Path
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "pyproject.toml").is_file():
        break
    repo_root = repo_root.parent
assert (repo_root / "pyproject.toml").is_file(), (
    "Couldn't find pyproject.toml. Are you running this notebook from inside the repo? (See 00.1 Section 2 for editor-specific kernel selection.)"
)
print(f"  ✓ Repo root: {repo_root}")

In [ ]:
# Hello-world end-to-end run on synthetic data. Builds a tiny dataset,
# trains for 1 epoch, and reports the final batch loss.
import torch
from torch.utils.data import DataLoader
from neural_data_decoding.data import SyntheticTrialDataset
from neural_data_decoding.data.dataset import collate_trials
from neural_data_decoding.models.classifier import MultiHeadClassifier

torch.manual_seed(0)
ds = SyntheticTrialDataset(
    num_sessions=2, trials_per_session=16, num_samples=8,
    num_features=4, num_classes_per_dim=[3], seed=0,
)
loader = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_trials)
model = MultiHeadClassifier(in_features=4, num_classes_per_dim=[3])
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for batch in loader:
    # collate_trials returns a dict with keys ('x', 'targets', 'metadata').
    x = batch['x']
    targets = batch['targets']
    # MultiHeadClassifier consumes (B, num_features); collapse W dim.
    x_flat = x.mean(dim=1)
    logits_list = model(x_flat)
    loss = torch.nn.functional.cross_entropy(logits_list[0], targets[:, 0])
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"  ✓ Hello-world run completed; final batch loss = {loss.item():.3f}")

If all five cells above show a `✓` check, you're done — head to **[00.3 The MATLAB → Python mental model](00.3_the_matlab_to_python_mental_model.ipynb)**.

## Section 4 — Hands-on exercise

Run the full test suite to confirm your install is healthy:

```bash
source .venv/bin/activate
python -m pytest -q
```

You should see something like `792 passed, 4 deselected in ~8s`. The 4 deselected tests are MATLAB-gated parity tests; they only run with `-m needs_matlab` and require a local MATLAB install.

If any test fails, capture the output and open an issue — your environment is almost certainly the cause, and we want to know about it.

## Section 5 — Common errors

### `command not found: python3`

Python isn't installed (or isn't on your PATH). On macOS install via [Homebrew](https://brew.sh): `brew install python@3.11`. On Linux: `sudo apt install python3.11 python3.11-venv`. On Windows: install from [python.org/downloads](https://www.python.org/downloads/) and make sure "Add to PATH" is checked.

### `ModuleNotFoundError: No module named 'neural_data_decoding'`

The venv isn't activated, or `pip install -e .` was never run, OR your editor is using the wrong Python interpreter. Diagnose with:

```python
import sys
print(sys.executable)
```

The path should end in `.venv/bin/python` (or `.venv\\Scripts\\python.exe` on Windows). If it doesn't, switch kernels:

* **VS Code:** open the command palette (`Cmd/Ctrl+Shift+P`) → **Python: Select Interpreter** → pick `.venv/bin/python`, then for the notebook **Select Kernel → Python Environments → `.venv/bin/python`**.
* **JupyterLab/Notebook:** `deactivate`, re-`source .venv/bin/activate`, and re-launch `jupyter lab` (or `jupyter notebook`) from the activated venv.

If the path is already correct, run `pip install -e .` from the repo root.

### `ModuleNotFoundError: No module named 'torch'`

Same root cause — torch is one of the project's dependencies, so `pip install -e .` should have pulled it in. If it didn't, run `pip install torch` explicitly.

### Editor can't find the venv kernel

* **VS Code:** if `.venv/bin/python` isn't in the Select Kernel list, run the **Python: Select Interpreter** command first (palette `Cmd/Ctrl+Shift+P`). Once interpreter is set, the kernel picker discovers it automatically.
* **JupyterLab:** if your venv kernel doesn't show up, register it explicitly: `python -m ipykernel install --user --name=ndd-venv`. Then pick `ndd-venv` from the kernel switcher.

### `pyright` errors in editor

Cosmetic; pyright is a static type checker your editor runs on its own — it is NOT installed by the project's dependencies, so editor and CI versions can disagree. Ignore editor-side errors unless `python -m pyright src/` (after `pip install pyright`) reproduces them in the terminal.

## Section 6 — Further reading

* [Python venv docs](https://docs.python.org/3/library/venv.html) — the canonical reference.
* [pip user guide](https://pip.pypa.io/en/stable/user_guide/) — install, upgrade, freeze, requirements.
* [VS Code Jupyter docs](https://code.visualstudio.com/docs/datascience/jupyter-notebooks) — for the VS Code path.
* [JupyterLab docs](https://jupyterlab.readthedocs.io/) — keyboard shortcuts, settings, extensions.
* [`CLAUDE.md`](../../CLAUDE.md) — project-specific shell conventions (`source .venv/bin/activate`, `python -m pytest`, etc.).

When you're done, head to **[00.3 The MATLAB → Python mental model](00.3_the_matlab_to_python_mental_model.ipynb)**.